# On the very first run, the pipeline performs a FULL load; all subsequent runs switch to INCREMENTAL mode.

In [0]:
import requests
import time
import json
from datetime import datetime, timezone
from typing import Optional, List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import Row

# ================= SECRETS =================

password = dbutils.secrets.get(scope="topdesk-scope", key="topdeskapi")
username = "topdeskapi"

# ================= CONFIG =================

CATALOG = "elixir"
BRONZE_SCHEMA = "brz"
META_SCHEMA = "meta"
BATCH_SIZE    = 10000

ENDPOINTS = [
    # {
    #     "name": "incidents",
    #     "url": "https://elixir.msf.org/tas/api/incidents",
    #     "fields": None,
    #     "incremental_field": "modificationDate",
    #     "bronze_table": "elixir.brz.topdesk_incidents_raw",
    #     "pagination": "OFFSET"
    # },

    # {
    #     "name": "time_registrations",
    #     "url": "https://elixir.msf.org/tas/api/incidents/timeregistrations",
    #     "fields":"timeSpent,operator.id,creationDate,parent.id,notes,modificationDate",
    #     "incremental_field": "modificationDate",
    #     "bronze_table": "elixir.brz.topdesk_timeregistrations_raw",
    #     "pagination": "NEXT"
         
    # },

    # {
    #     "name": "priorities",
    #     "url": "https://elixir.msf.org/tas/api/incidents/priorities",
    #     "fields": None,
    #     "incremental_field": None,
    #     "bronze_table": "elixir.brz.topdesk_incident_priorities_raw",
    #     "pagination": "SINGLE"

    # },

    # {
    #     "name": "statuses",
    #     "url": "https://elixir.msf.org/tas/api/incidents/statuses",
    #     "fields": None,
    #     "incremental_field": None,
    #     "bronze_table": "elixir.brz.topdesk_incident_statuses_raw",
    #     "pagination": "SINGLE"

    # },

    # {
    #     "name": "urgencies",
    #     "url": "https://elixir.msf.org/tas/api/incidents/urgencies",
    #     "fields": None,
    #     "incremental_field": None,
    #     "bronze_table": "elixir.brz.topdesk_incident_urgenciess_raw",
    #     "pagination": "SINGLE"
    # },
    
    # {
    #     "name": "categories",
    #     "url": "https://elixir.msf.org/tas/api/incidents/categories",
    #     "fields":None,
    #     "incremental_field": None,
    #     "bronze_table": "elixir.brz.topdesk_incident_categories_raw",
    #     "pagination": "SINGLE"
    # },
     
    # {
    #     "name": "subcategories",
    #     "url": "https://elixir.msf.org/tas/api/incidents/subcategories",
    #     "fields": None,
    #     "incremental_field": None,
    #     "bronze_table": "elixir.brz.topdesk_incident_subcategories_raw",
    #     "pagination": "SINGLE"
    # },

    # {
    #     "name": "closure_codes",
    #     "url": "https://elixir.msf.org/tas/api/incidents/closure_codes",
    #     "fields": None,
    #     "incremental_field": None,
    #     "bronze_table": "elixir.brz.topdesk_incident_closurecodes_raw",
    #     "pagination": "SINGLE"
    # },
    

    # {
    #     "name": "branches",
    #     "url": "https://elixir.msf.org/tas/api/branches",
    #     "fields": None,
    #     "incremental_field": None,
    #     "bronze_table": "elixir.brz.topdesk_branches_raw",
    #     "pagination": "SINGLE"
        
    # },

    # {
    #     "name": "budgetholders",
    #     "url": "https://elixir.msf.org/tas/api/budgetholders",
    #     "fields": None,
    #     "incremental_field": None,
    #     "bronze_table": "elixir.brz.topdesk_budgetholders_raw",
    #     "pagination": "SINGLE"
    # },

    # {
    #     "name": "operators",
    #     "url": "https://elixir.msf.org/tas/api/operators",
    #     "fields": None,
    #     "incremental_field": None,
    #     "bronze_table": "elixir.brz.topdesk_operators_raw",
    #     "pagination": "OFFSET"
    # },
    
    # {
    #     "name": "operatorgroup",
    #     "url": "https://elixir.msf.org/tas/api/operatorgroups",
    #     "fields": None,
    #     "incremental_field": None,
    #     "bronze_table": "elixir.brz.topdesk_operatorgroup_raw",
    #     "pagination": "OFFSET"
    # },

    {
        "name": "impacts",
        "url": "https://elixir.msf.org/tas/api/changes/impacts",
        "fields": None,
        "incremental_field": None,
        "bronze_table": "elixir.brz.topdesk_changes_impacts_raw",
        "pagination": "SINGLE"
    },

    {
        "name": "operatorchanges",
        "url": "https://elixir.msf.org/tas/api/operatorChanges",
        "fields": "id,incident.id,changeType,status,category.id,category.name,subcategory.id,processingStatus",
        "incremental_field": "modificationDate",
        "bronze_table": "elixir.brz.topdesk_operator_changes_raw",
        "pagination": "NEXT"
    },

    # {
    #     "name": "changeactivities",
    #     "url": "https://elixir.msf.org/tas/api/operatorChangeActivities",
    #     "fields": "timeSpent,id,category.id,lastModificationDate",
    #     "incremental_field": "lastModificationDate",
    #     "bronze_table": "elixir.brz.topdesk_operator_changeActivities_raw",
    #     "pagination": "NEXT"
    # }
 ]




AUTH = (username, password)

PAGE_SIZE = 100
MAX_WORKERS = 10
MAX_RETRIES = 3
REQUEST_TIMEOUT = 30

LOAD_TYPE = "FULL"   # "FULL" or "INCREMENTAL"
STATE_TABLE  = f"{CATALOG}.{META_SCHEMA}.pipeline_state"

PIPELINE_NAME = "topdesk_ingestion"


In [0]:
def format_ts(dt: datetime) -> str:
    # Format datetime to ISO 8601 string with milliseconds and Zulu (UTC) suffix
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"



def parse_ts(value: Optional[str]) -> Optional[datetime]:
    if not value:
        return None

    # normalize Z → +00:00 so Python can parse it
    if value.endswith("Z"):
        value = value.replace("Z", "+00:00")

    return datetime.fromisoformat(value)
    
def build_url(endpoint, query):
    base = endpoint["url"]

    params = []

    if endpoint.get("fields"):
        params.append(f"fields={endpoint['fields']}")

    if query:
        params.append(f"query={query}")

    if params:
        return base + "?" + "&".join(params)

    return base

# ============================================================
# STATE MANAGEMENT
# ============================================================

def get_last_run(endpoint_name: str):
    if LOAD_TYPE == "FULL":
        return None

    rows = spark.sql(f"""
        SELECT last_run
        FROM {STATE_TABLE}
        WHERE pipeline = '{PIPELINE_NAME}'
        AND endpoint = '{endpoint_name}'
        LIMIT 1
    """).collect()

    return rows[0]["last_run"] if rows else None


def ensure_state_table_exists():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {STATE_TABLE} (
            pipeline STRING,
            endpoint STRING,
            last_run STRING
        )
        USING DELTA
    """)



def update_last_run(endpoint_name: str, ts: str):
    spark.sql(f"""
        MERGE INTO {STATE_TABLE} t
        USING (
            SELECT '{PIPELINE_NAME}' AS pipeline,
                   '{endpoint_name}' AS endpoint,
                   '{ts}' AS last_run
        ) s
        ON t.pipeline = s.pipeline AND t.endpoint = s.endpoint
        WHEN MATCHED THEN UPDATE SET t.last_run = s.last_run
        WHEN NOT MATCHED THEN INSERT *
    """)


# ============================================================
# QUERY
# ============================================================


def build_query(last_run, endpoint):
    inc_field = endpoint.get("incremental_field")

    if LOAD_TYPE == "FULL" or not last_run or not inc_field:
        return None

    #return f"{inc_field}>{last_run}"
    return f"{inc_field}>'{last_run}'"




# ============================================================
# OFFSET PAGINATION (PARALLEL)
# ============================================================


def fetch_page(session, endpoint, start, query):
    params = {
        "start": start,
        "page_size": PAGE_SIZE
    }

    if query:
        params["query"] = query

    if endpoint.get("fields"):
        params["fields"] = endpoint["fields"]

    resp = session.get(endpoint["url"], params=params, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    if not resp.text.strip():
       return []

    return resp.json()
    #return resp.json()

def stream_endpoint_to_bronze(endpoint, query):
    endpoint_name = endpoint["name"]
    inc_field = endpoint.get("incremental_field")
    pagination_type = endpoint["pagination"]
    
    if pagination_type not in {"SINGLE", "OFFSET", "NEXT"}:
        raise Exception(f"Invalid pagination type: {pagination_type}")

    print(f"[INFO] {endpoint_name} pagination: {pagination_type}")

    total = 0
    batch = []
    max_ts = None

    # =========================================================
    # SINGLE (NO PAGINATION)
    # =========================================================
    if pagination_type == "SINGLE":
        url = build_url(endpoint, query)

        with requests.Session() as session:
            session.auth = AUTH
            resp = session.get(url, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()

            payload = resp.json()
            # Handle both list and {"results:[.....]"}
            if isinstance(payload, dict):
                payload = payload.get("results", [])
            else:
                payload = payload

            for rec in payload:
                # track max timestamp safely
                # if LOAD_TYPE != "FULL" and inc_field:
                if inc_field:
                    value = rec.get(inc_field)

                    if value:
                        parsed = parse_ts(value)

                        if parsed and (max_ts is None or parsed > max_ts):
                            max_ts = parsed
                batch.append(rec)
                total += 1


    # =========================================================
    # NEXT PAGINATION
    # =========================================================
    elif pagination_type == "NEXT":
        url = build_url(endpoint, query)

        with requests.Session() as session:
            session.auth = AUTH

            while url:
                resp = session.get(url, timeout=REQUEST_TIMEOUT)
                resp.raise_for_status()

                payload = resp.json()
                results = payload.get("results", [])

                for rec in results:
                    # track max timestamp safely
                    # if LOAD_TYPE != "FULL" and inc_field:
                    if inc_field:
                        value = rec.get(inc_field)

                        if value:
                            parsed = parse_ts(value)

                            if parsed and (max_ts is None or parsed > max_ts):
                                max_ts = parsed
                        # mod = rec.get(inc_field)
                        # if mod and (max_ts is None or mod > max_ts):
                        #     max_ts = mod

                    batch.append(rec)
                    total += 1

                if len(batch) >= BATCH_SIZE:
                    write_to_bronze(batch, endpoint)
                    batch.clear()

                print(f"[INFO] {endpoint_name} fetched={total}")

                url = payload.get("next")

    # =========================================================
    # OFFSET PAGINATION
    # =========================================================
    else: # OFFSET
        next_start = 0

        with requests.Session() as session:
            session.auth = AUTH

            while True:
                data = fetch_page(session, endpoint, next_start, query)

                if not data:
                    break

                for rec in data:
                    # if LOAD_TYPE != "FULL" and inc_field:
                    if inc_field:
                        value = rec.get(inc_field)

                        if value:
                            parsed = parse_ts(value)

                            if parsed and (max_ts is None or parsed > max_ts):
                                max_ts = parsed
                        # mod = rec.get(inc_field)
                        # if mod and (max_ts is None or mod > max_ts):
                        #     max_ts = mod

                    batch.append(rec)
                    total += 1

                if len(batch) >= BATCH_SIZE:
                    write_to_bronze(batch, endpoint)
                    batch.clear()

                if len(data) < PAGE_SIZE:
                    break

                next_start += PAGE_SIZE

                print(f"[INFO] {endpoint_name} fetched={total}")

    if batch:
        write_to_bronze(batch, endpoint)

    return total, format_ts(max_ts) if max_ts else None


def write_to_bronze(incidents: List[Dict], endpoint) -> None:
    """Write raw incidents into bronze Delta table."""
    bronze_table = endpoint["bronze_table"]
    now = datetime.now(timezone.utc).isoformat()

    rows = [
        Row(
            raw_json=json.dumps(rec),
            ingestion_time=now
        )
        for rec in incidents
    ]

    df = spark.createDataFrame(rows)

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_table)
    )

    print(f"[INFO] Written {len(incidents)} records to {bronze_table}")


def main():
    """
    ============================================================
    TOPDESK INGESTION PIPELINE
    ============================================================

    This pipeline ingests multiple TOPdesk API endpoints into
    Bronze Delta tables.

    Features:
    - Supports FULL and INCREMENTAL loads
    - Handles multiple endpoints with different schemas
    - Supports both OFFSET and NEXT pagination styles
    - Writes data in batches for performance and stability
    - Tracks incremental state per endpoint

    Flow per endpoint:
        1. Read last_run from state table
        2. Build query filter (if incremental)
        3. Fetch data from API
        4. Write to Bronze in batches
        5. Update last_run checkpoint

    ============================================================
    """
    print(f"[INFO] Running pipeline in {LOAD_TYPE} mode")

    ensure_state_table_exists()

    for endpoint in ENDPOINTS:
        print("=" * 60)

        endpoint_name = endpoint["name"]
        print(f"[PIPELINE] {endpoint_name}")

        last_run = get_last_run(endpoint_name)

        query = build_query(last_run, endpoint)
        if query:
            print(f"[INFO] Filter: {query}")
        else:
            print("[INFO] FULL load (no filter)")

        total, max_ts = stream_endpoint_to_bronze(endpoint, query)

        print(f"[DONE] {endpoint_name} -> {total} records")

        # update state only for incremental endpoints
        if endpoint.get("incremental_field") and max_ts:
            update_last_run(endpoint_name, max_ts)
            print(f"[INFO] Updated last_run → {max_ts}")


In [0]:
if __name__ == "__main__":
    main()

In [0]:
%sql

UPDATE elixir.meta.pipeline_state
SET last_run = '2026-02-26T14:00:00.000Z'
WHERE pipeline = 'topdesk_ingestion'
AND endpoint = 'time_registrations'

In [0]:

url = "https://elixir.msf.org/tas/api/operatorChanges"
#url = "https://elixir.msf.org/tas/api/incidents/closure_codes"

with requests.Session() as s:
    s.auth = AUTH
    r = s.get(url)
    print(r.status_code)
    print(r.text)
    print(r.json())

In [0]:
url = "https://elixir.msf.org/tas/api/incidents/closure_codes"
with requests.Session() as session:
    session.auth = AUTH

    r1 = session.get(
        url,
        params={"start": 0, "page_size": 1},
        timeout=REQUEST_TIMEOUT
    )
    r1.raise_for_status()
    data1 = r1.json()

    r2 = session.get(
        url,
        params={"start": 1, "page_size": 100},
        timeout=REQUEST_TIMEOUT
    )
    r2.raise_for_status()
    data2 = r2.json()

print("page_size=1 ->", len(data1))
print("page_size=100 ->", len(data2))
print("First IDs equal?:", data1[0]["id"] == data2[0]["id"])

In [0]:
import json
import logging
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple, Any

import requests
from pyspark.sql import Row
from pyspark.sql.functions import col
from tenacity import retry, stop_after_attempt, wait_exponential

# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("topdesk_pipeline")


# ============================================================
# CONFIG
# ============================================================

@dataclass
class PipelineConfig:
    load_type: str  = "INCREMENTAL"  # "FULL" or "INCREMENTAL"
    pipeline_name: str = "topdesk_ingestion"  # logical pipeline identifier
    state_table: str  = f"{CATALOG}.{META_SCHEMA}.pipeline_state"   # delta table storing checkpoints
    page_size: int = 1000          # API page size
    request_timeout: int = 60     # seconds per request
    batch_size: int = 25000           # write to bronze every N records
    auth: Any = AUTH                # requests auth object (BasicAuth, token, etc.)


# ============================================================
# URL / QUERY
# ============================================================

def build_url(endpoint: Dict[str, Any], query: Optional[str]) -> str:
    base = endpoint["url"]
    params = []

    if endpoint.get("fields"):
        params.append(f"fields={endpoint['fields']}")

    if query:
        params.append(f"query={query}")

    return base + ("?" + "&".join(params) if params else "")


def build_query(last_run: Optional[str], endpoint: Dict[str, Any], cfg: PipelineConfig) -> Optional[str]:
    inc_field = endpoint.get("incremental_field")

    if cfg.load_type == "FULL" or not last_run or not inc_field:
        return None

    return f"{inc_field}>{last_run}"


# ============================================================
# STATE MANAGEMENT (SAFE - no SQL injection)
# ============================================================

def ensure_state_table_exists(cfg: PipelineConfig):
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {cfg.state_table} (
            pipeline STRING,
            endpoint STRING,
            last_run STRING
        )
        USING DELTA
    """)


def get_last_run(endpoint_name: str, cfg: PipelineConfig) -> Optional[str]:
    if cfg.load_type == "FULL":
        return None

    df = (
        spark.table(cfg.state_table)
        .filter(
            (col("pipeline") == cfg.pipeline_name) &
            (col("endpoint") == endpoint_name)
        )
        .orderBy(col("last_run").desc())
        .limit(1)
    )

    rows = df.collect()
    return rows[0]["last_run"] if rows else None


def update_last_run(endpoint_name: str, ts: str, cfg: PipelineConfig):
    spark.sql(f"""
        MERGE INTO {cfg.state_table} t
        USING (
            SELECT '{cfg.pipeline_name}' AS pipeline,
                   '{endpoint_name}' AS endpoint,
                   '{ts}' AS last_run
        ) s
        ON t.pipeline = s.pipeline AND t.endpoint = s.endpoint
        WHEN MATCHED THEN UPDATE SET t.last_run = s.last_run
        WHEN NOT MATCHED THEN INSERT *
    """)


# ============================================================
# RETRYABLE HTTP
# ============================================================

@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=1, min=1, max=10))
def http_get(session: requests.Session, url: str, **kwargs):
    resp = session.get(url, **kwargs)
    resp.raise_for_status()
    return resp


# ============================================================
# PAGINATION HELPERS
# ============================================================

def parse_ts(value: Optional[str]) -> Optional[datetime]:
    if not value:
        return None
    return datetime.fromisoformat(value)


def update_max_ts(rec: Dict, inc_field: str, max_ts: Optional[datetime]) -> Optional[datetime]:
    value = rec.get(inc_field)
    if not value:
        return max_ts

    parsed = parse_ts(value)

    if parsed and (max_ts is None or parsed > max_ts):
        return parsed

    return max_ts


def fetch_page(session, endpoint, start, query, cfg: PipelineConfig):
    params = {
        "start": start,
        "page_size": cfg.page_size
    }

    if query:
        params["query"] = query

    if endpoint.get("fields"):
        params["fields"] = endpoint["fields"]

    resp = http_get(
        session,
        endpoint["url"],
        params=params,
        timeout=cfg.request_timeout
    )

    return resp.json()


# ============================================================
# BRONZE WRITE
# ============================================================

def write_to_bronze(records: List[Dict], endpoint: Dict[str, Any]) -> None:
    bronze_table = endpoint["bronze_table"]
    now = datetime.now(timezone.utc).isoformat()

    rows = [
        Row(
            raw_json=json.dumps(rec),
            ingestion_time=now
        )
        for rec in records
    ]

    df = spark.createDataFrame(rows)

    (
        df.write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_table)
    )

    logger.info(f"Written {len(records)} records to {bronze_table}")


# ============================================================
# STREAMING INGEST
# ============================================================

def stream_endpoint_to_bronze(
    endpoint: Dict[str, Any],
    query: Optional[str],
    cfg: PipelineConfig
) -> Tuple[int, Optional[str]]:

    endpoint_name = endpoint["name"]
    inc_field = endpoint.get("incremental_field")

    total = 0
    batch: List[Dict] = []
    max_ts: Optional[datetime] = None

    # --- detect pagination type ---
    with requests.Session() as session:
        session.auth = cfg.auth
        probe = http_get(
            session,
            endpoint["url"],
            params={"start": 0, "page_size": 1},
            timeout=cfg.request_timeout
        )
        sample = probe.json()

    is_next = isinstance(sample, dict) and "next" in sample
    logger.info(f"{endpoint_name} pagination: {'NEXT' if is_next else 'OFFSET'}")

    # =========================================================
    # NEXT PAGINATION
    # =========================================================
    if is_next:
        url = build_url(endpoint, query)

        with requests.Session() as session:
            session.auth = cfg.auth

            while url:
                payload = http_get(session, url, timeout=cfg.request_timeout).json()

                results = payload.get("results", [])

                for rec in results:
                    if cfg.load_type != "FULL" and inc_field:
                        max_ts = update_max_ts(rec, inc_field, max_ts)

                    batch.append(rec)
                    total += 1

                if len(batch) >= cfg.batch_size:
                    write_to_bronze(batch, endpoint)
                    batch.clear()

                logger.info(f"{endpoint_name} fetched={total}")
                url = payload.get("next")

                time.sleep(0.05)  # light throttle

    # =========================================================
    # OFFSET PAGINATION
    # =========================================================
    else:
        next_start = 0

        with requests.Session() as session:
            session.auth = cfg.auth

            while True:
                data = fetch_page(session, endpoint, next_start, query, cfg)

                if not data:
                    break

                for rec in data:
                    if cfg.load_type != "FULL" and inc_field:
                        max_ts = update_max_ts(rec, inc_field, max_ts)

                    batch.append(rec)
                    total += 1

                if len(batch) >= cfg.batch_size:
                    write_to_bronze(batch, endpoint)
                    batch.clear()

                if len(data) < cfg.page_size:
                    break

                next_start += cfg.page_size
                logger.info(f"{endpoint_name} fetched={total}")

                time.sleep(0.05)

    if batch:
        write_to_bronze(batch, endpoint)

    return total, max_ts.isoformat() if max_ts else None


# ============================================================
# MAIN
# ============================================================

def main():
    logger.info(f"Running pipeline in {LOAD_TYPE} mode")

    cfg = PipelineConfig(
        load_type=LOAD_TYPE,
        pipeline_name=PIPELINE_NAME,
        state_table=STATE_TABLE,
        auth=AUTH,
        page_size=PAGE_SIZE,
        request_timeout=REQUEST_TIMEOUT
    )

    ensure_state_table_exists(cfg)

    for endpoint in ENDPOINTS:
        logger.info("=" * 60)
        endpoint_name = endpoint["name"]
        logger.info(f"PIPELINE {endpoint_name}")

        last_run = get_last_run(endpoint_name, cfg)
        query = build_query(last_run, endpoint, cfg)

        total, max_ts = stream_endpoint_to_bronze(endpoint, query, cfg)

        logger.info(f"DONE {endpoint_name} -> {total} records")

        if endpoint.get("incremental_field") and max_ts:
            update_last_run(endpoint_name, max_ts, cfg)

In [0]:
if __name__ == "__main__":
    main()

In [0]:
test_url = build_url(ENDPOINTS[0], "modificationDate>2026-02-26T14:00:00.000Z")
print(test_url)

In [0]:
for endpoint in ENDPOINTS:
    print(endpoint)
    